In [2]:
! pip install numpy

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [4]:
import os
import torch
import timm
import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import general_fcns as gf  # Your helper with slide_at_magnification

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load GigaPath tile encoder
tile_encoder = timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True)
tile_encoder.eval().to(device)

# Define image transform (from GigaPath guide)
transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Define output folder
output_folder = './features'
os.makedirs(output_folder, exist_ok=True)

# CSV path and skip list
csv_file_path = os.path.join(os.getcwd(), 'filtered_metadata_full_path.csv')
skip = ["D:/Digital_path_unzipped/SZMC1050237_pdl1.ndpi", "D:/Digital_path/DIG_PAT_1727366215.ndpi"]

# Load list of image paths
df = pd.read_csv(csv_file_path)
image_paths = df.iloc[:, 0].dropna().tolist()
image_paths = [p for p in image_paths if p not in skip]

valid_exts = ('.png', '.jpg', '.jpeg', '.tif', '.svs', '.ndpi')

# Process each image in the filtered list
for img_path in tqdm(image_paths):
    if not img_path.lower().endswith(valid_exts):
        continue

    img_file = os.path.basename(img_path)
    base_name = os.path.splitext(img_file)[0]
    out_path = os.path.join(output_folder, f"{base_name}.pt")

    print(f"Processing: {img_file}")

    try:
        # Load and preprocess image
        if img_path.lower().endswith('.svs') or img_path.lower().endswith('.ndpi') or img_path.lower().endswith('.tif'):
            slide = gf.slide_at_magnification(img_path, magnification_params={'magnification': 10})
            image = Image.fromarray(slide)
        else:
            image = Image.open(img_path).convert('RGB')

        image_tensor = transform(image).unsqueeze(0).to(device)

        # Extract features
        with torch.no_grad():
            patch_feature = tile_encoder(image_tensor).squeeze().cpu()

        # Save features
        torch.save(patch_feature, out_path)

    except Exception as e:
        print(f"Failed to process {img_file}: {e}")


  0%|          | 0/469 [00:00<?, ?it/s]

Processing: DIG_PAT_1701101606.tif


  0%|          | 1/469 [00:32<4:15:40, 32.78s/it]

Processing: DIG_PAT_1701103165.tif


  0%|          | 2/469 [11:30<51:58:15, 400.63s/it]

...

100%|█████████▉| 467/469 [37:44:57<05:46, 173.44s/it]

Processing: DIG_PAT_1727367771.ndpi


100%|█████████▉| 468/469 [37:51:00<03:50, 230.19s/it]

Processing: DIG_PAT_1728846413.tif


100%|██████████| 469/469 [37:51:36<00:00, 290.61s/it]


In [ ]:
import os
import torch
import timm
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import general_fcns as gf  # Make sure this is implemented with slide_at_magnification

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load GigaPath tile encoder
tile_encoder = timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True)
tile_encoder.eval().to(device)

# Define image transform (from GigaPath guide)
transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [ ]:
# Define folders
image_folder = r"D:/Digital_path" 
output_folder = './features'
os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Process each image
for img_file in tqdm(os.listdir(image_folder)):
    if not img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.svs')):
        continue

    img_path = os.path.join(image_folder, img_file)
    base_name = os.path.splitext(img_file)[0]
    out_path = os.path.join(output_folder, f"{base_name}.pt")

    print(f"Processing: {img_file}")

    try:
        # Load and preprocess image
        if img_file.lower().endswith('.svs'):
            slide = gf.slide_at_magnification(img_path, magnification_params={'magnification': 1})
            image = Image.fromarray(slide)
        else:
            image = Image.open(img_path).convert('RGB')

        image_tensor = transform(image).unsqueeze(0).to(device)

        # Extract features
        with torch.no_grad():
            patch_feature = tile_encoder(image_tensor).squeeze().cpu()

        # Save features
        torch.save(patch_feature, out_path)

    except Exception as e:
        print(f"Failed to process {img_file}: {e}")

  0%|          | 0/8 [00:00<?, ?it/s]

Processing: DIG_PAT_1697003050.svs


 12%|█▎        | 1/8 [00:01<00:11,  1.62s/it]

Processing: DIG_PAT_1697003063.svs


 25%|██▌       | 2/8 [00:03<00:10,  1.68s/it]

...

 75%|███████▌  | 6/8 [00:09<00:03,  1.88s/it]

Processing: DIG_PAT_1697003266.svs


 88%|████████▊ | 7/8 [00:11<00:01,  1.76s/it]

Processing: DIG_PAT_1697003309.svs


100%|██████████| 8/8 [00:11<00:00,  1.47s/it]
